## Pandas: Combining Datasets

In [1]:
#import and auxiliary functions
import pandas as pd
import numpy as np

def make_df(cols, ind):
    """Quickly make a DataFrame"""
    data = {c: [str(c) + str(i) for i in ind] for c in cols}
    return pd.DataFrame(data, ind)

# example DataFrame
make_df('ABC', range(3))

,A,B,C
0,A0,B0,C0
1,A1,B1,C1
2,A2,B2,C2


#### Concatenation using concat

In [2]:
x = [[1,2],
    [3,4]]
np.concatenate([x,x], axis=1)

array([[1, 2, 1, 2],
       [3, 4, 3, 4]])

In [ ]:
#pd.concat
#pd.concat(objs, axis=0, join='outer', join_axes=None,
#          ignore_index=False,keys=None, levels=None,
#          names=None, verify_integrity=False, copy=True)

In [7]:
pd.concat?

In [4]:
# series
ser1 = pd.Series(['A', 'B', 'C'], index=[1, 2, 3])
ser2 = pd.Series(['D', 'E', 'F'], index=[4, 5, 6])
pd.concat([ser1, ser2])

,0
1,A
2,B
3,C
4,D
5,E
6,F


In [5]:
#DataFrames
df1 = make_df('AB', [1, 2])
df2 = make_df('AB', [3, 4])
print(df1,'\n'); print(df2,'\n')
print(pd.concat([df1, df2]), '\n')
print(pd.concat([df1, df2], axis=1)) #along the column

    A   B
1  A1  B1
2  A2  B2 

    A   B
3  A3  B3
4  A4  B4 

    A   B
1  A1  B1
2  A2  B2
3  A3  B3
4  A4  B4 

     A    B    A    B
1   A1   B1  NaN  NaN
2   A2   B2  NaN  NaN
3  NaN  NaN   A3   B3
4  NaN  NaN   A4   B4


In [6]:
print(pd.concat([df1, df2], axis=1, join="inner"))

Empty DataFrame
Columns: [A, B, A, B]
Index: []


In [8]:
##deal with duplicate indices
print(pd.concat([df1, df1]), '\n')

    A   B
1  A1  B1
2  A2  B2
1  A1  B1
2  A2  B2 



In [9]:
print(pd.concat([df1, df1], verify_integrity=True))

ValueError: Indexes have overlapping values: Index([1, 2], dtype='int64')

In [10]:
# ignore: reordered
print(pd.concat([df2, df1]), '\n')
print(pd.concat([df2, df1], ignore_index=True))

    A   B
3  A3  B3
4  A4  B4
1  A1  B1
2  A2  B2 

    A   B
0  A3  B3
1  A4  B4
2  A1  B1
3  A2  B2


In [11]:
# add keys indicating sources
df3 = df1
df4 = pd.concat([df1, df3], keys=['df1','df3'])
print(df4)

        A   B
df1 1  A1  B1
    2  A2  B2
df3 1  A1  B1
    2  A2  B2


In [12]:
df4.index

MultiIndex([('df1', 1),
            ('df1', 2),
            ('df3', 1),
            ('df3', 2)],
           )

In [13]:
# union/intersection of the input columns
df5 = make_df('ABC', [1, 2])
df6 = make_df('BCD', [3, 4])
print(df5,'\n'); print(df6,'\n')
print(pd.concat([df5, df6]),'\n')

#intersection: inner
print(pd.concat([df5, df6],join='inner'))

    A   B   C
1  A1  B1  C1
2  A2  B2  C2 

    B   C   D
3  B3  C3  D3
4  B4  C4  D4 

     A   B   C    D
1   A1  B1  C1  NaN
2   A2  B2  C2  NaN
3  NaN  B3  C3   D3
4  NaN  B4  C4   D4 

    B   C
1  B1  C1
2  B2  C2
3  B3  C3
4  B4  C4


#### Concatenation using append

In [14]:
print(df2.append(df1))

AttributeError: 'DataFrame' object has no attribute 'append'

#### Merge

In [15]:
#one to one join
df1 = pd.DataFrame({'employee': ['Bob', 'Jake', 'Lisa', 'Sue'],
                    'group': ['Accounting', 'Engineering', 'Engineering', 'HR']})
df2 = pd.DataFrame({'employee': ['Lisa', 'Bob', 'Jake', 'Sue'],
                    'hire_date': [2004, 2008, 2012, 2014]})
print(df1,'\n'); print(df2,'\n')

  employee        group
0      Bob   Accounting
1     Jake  Engineering
2     Lisa  Engineering
3      Sue           HR 

  employee  hire_date
0     Lisa       2004
1      Bob       2008
2     Jake       2012
3      Sue       2014 



In [16]:
df3 = pd.merge(df1, df2)
print(df3,'\n')

  employee        group  hire_date
0      Bob   Accounting       2008
1     Jake  Engineering       2012
2     Lisa  Engineering       2004
3      Sue           HR       2014 



In [17]:
#many to one join
df4 = pd.DataFrame({'group': ['Accounting', 'Engineering', 'HR'],
                    'supervisor': ['Carly', 'Guido', 'Steve']})
print(df3,'\n'); print(df4,'\n');
#additional column with the “supervisor” information,
# information repeated as required by the inputs
print(pd.merge(df3, df4))

  employee        group  hire_date
0      Bob   Accounting       2008
1     Jake  Engineering       2012
2     Lisa  Engineering       2004
3      Sue           HR       2014 

         group supervisor
0   Accounting      Carly
1  Engineering      Guido
2           HR      Steve 

  employee        group  hire_date supervisor
0      Bob   Accounting       2008      Carly
1     Jake  Engineering       2012      Guido
2     Lisa  Engineering       2004      Guido
3      Sue           HR       2014      Steve


In [18]:
#many to many join
df5 = pd.DataFrame({'group': ['Accounting', 'Accounting',
                              'Engineering', 'Engineering', 'HR', 'HR'],
                    'skills': ['math', 'spreadsheets', 'coding', 'linux',
                               'spreadsheets', 'organization']})
print(df1,'\n'); print(df5,'\n')
print(pd.merge(df1, df5))
#group correspond to two skills, thus two rows per employee

  employee        group
0      Bob   Accounting
1     Jake  Engineering
2     Lisa  Engineering
3      Sue           HR 

         group        skills
0   Accounting          math
1   Accounting  spreadsheets
2  Engineering        coding
3  Engineering         linux
4           HR  spreadsheets
5           HR  organization 

  employee        group        skills
0      Bob   Accounting          math
1      Bob   Accounting  spreadsheets
2     Jake  Engineering        coding
3     Jake  Engineering         linux
4     Lisa  Engineering        coding
5     Lisa  Engineering         linux
6      Sue           HR  spreadsheets
7      Sue           HR  organization


In [19]:
# specify the merge key
print(df1,'\n'); print(df2,'\n'); print(pd.merge(df1, df2, on='employee'))

  employee        group
0      Bob   Accounting
1     Jake  Engineering
2     Lisa  Engineering
3      Sue           HR 

  employee  hire_date
0     Lisa       2004
1      Bob       2008
2     Jake       2012
3      Sue       2014 

  employee        group  hire_date
0      Bob   Accounting       2008
1     Jake  Engineering       2012
2     Lisa  Engineering       2004
3      Sue           HR       2014


In [20]:
pd.concat([df1,df2],axis=1) #undesired

,employee,group,employee,hire_date
0,Bob,Accounting,Lisa,2004
1,Jake,Engineering,Bob,2008
2,Lisa,Engineering,Jake,2012
3,Sue,HR,Sue,2014


In [21]:
# different keys for different datasets
df3 = pd.DataFrame({'name': ['Bob', 'Jake', 'Lisa', 'Sue'],
                    'salary': [70000, 80000, 120000, 90000]})
print(df1,'\n'); print(df3,'\n');
print(pd.merge(df1, df3, left_on="employee", right_on="name"))

  employee        group
0      Bob   Accounting
1     Jake  Engineering
2     Lisa  Engineering
3      Sue           HR 

   name  salary
0   Bob   70000
1  Jake   80000
2  Lisa  120000
3   Sue   90000 

  employee        group  name  salary
0      Bob   Accounting   Bob   70000
1     Jake  Engineering  Jake   80000
2     Lisa  Engineering  Lisa  120000
3      Sue           HR   Sue   90000


In [22]:
# drop the duplicated one
pd.merge(df1, df3,
         left_on="employee", right_on="name").drop('name', axis=1)

,employee,group,salary
0,Bob,Accounting,70000
1,Jake,Engineering,80000
2,Lisa,Engineering,120000
3,Sue,HR,90000


In [23]:
#index merge: employee as the row index this name
df1a = df1.set_index('employee')
df2a = df2.set_index('employee')
print(df1a,'\n'); print(df2a,'\n')

                group
employee             
Bob        Accounting
Jake      Engineering
Lisa      Engineering
Sue                HR 

          hire_date
employee           
Lisa           2004
Bob            2008
Jake           2012
Sue            2014 



In [24]:
#then merge using indices
print(pd.merge(df1a, df2a, left_index=True, right_index=True))

                group  hire_date
employee                        
Bob        Accounting       2008
Jake      Engineering       2012
Lisa      Engineering       2004
Sue                HR       2014


In [25]:
# join(): merge using indices by default
print(df1a.join(df2a))

                group  hire_date
employee                        
Bob        Accounting       2008
Jake      Engineering       2012
Lisa      Engineering       2004
Sue                HR       2014


In [26]:
# mixed of index and column
print(pd.merge(df1a, df3, left_index=True, right_on='name'))

         group  name  salary
0   Accounting   Bob   70000
1  Engineering  Jake   80000
2  Engineering  Lisa  120000
3           HR   Sue   90000


#### Row-wise consideration

In [27]:
#example
df6 = pd.DataFrame({'name': ['Peter', 'Paul', 'Mary'],
                    'food': ['fish', 'beans', 'bread']},
                   columns=['name', 'food'])
df7 = pd.DataFrame({'name': ['Mary', 'Joseph'],
                    'drink': ['wine', 'beer']},
                   columns=['name', 'drink'])
print(df6,'\n'); print(df7,'\n');

    name   food
0  Peter   fish
1   Paul  beans
2   Mary  bread 

     name drink
0    Mary  wine
1  Joseph  beer 



In [28]:
print(pd.merge(df6, df7))
#equivalent
print(pd.merge(df6, df7, how='inner'))

   name   food drink
0  Mary  bread  wine
   name   food drink
0  Mary  bread  wine


In [29]:
#how argument
print(pd.merge(df6, df7, how='outer'),'\n')

print(pd.merge(df6, df7, how='left'))

     name   food drink
0  Joseph    NaN  beer
1    Mary  bread  wine
2    Paul  beans   NaN
3   Peter   fish   NaN 

    name   food drink
0  Peter   fish   NaN
1   Paul  beans   NaN
2   Mary  bread  wine


#### Overlapping column names

In [30]:
df8 = pd.DataFrame({'name': ['Bob', 'Jake', 'Lisa', 'Sue'],
                    'rank': [1, 2, 3, 4]})
df9 = pd.DataFrame({'name': ['Bob', 'Jake', 'Lisa', 'Sue'],
                    'rank': [3, 1, 4, 2]})
print(df8,'\n'); print(df9,'\n')

   name  rank
0   Bob     1
1  Jake     2
2  Lisa     3
3   Sue     4 

   name  rank
0   Bob     3
1  Jake     1
2  Lisa     4
3   Sue     2 



In [32]:
print(pd.merge(df8, df9, on="name"),'\n')
print(pd.merge(df8, df9, on="name", suffixes=["_2020", "_2024"]))

   name  rank_x  rank_y
0   Bob       1       3
1  Jake       2       1
2  Lisa       3       4
3   Sue       4       2 

   name  rank_2020  rank_2024
0   Bob          1          3
1  Jake          2          1
2  Lisa          3          4
3   Sue          4          2


##### Exercise 1
Append the information from an existing Pandas series to an existing DataFrame, reindex the row index, and display the combined data.

In [33]:
import pandas as pd
student_data1  = pd.DataFrame({
        'student_id': ['S1', 'S2', 'S3', 'S4', 'S5'],
         'name': ['Danniella Fenton', 'Ryder Storey', 'Bryce Jensen', 'Ed Bernal', 'Kwame Morin'],
        'marks': [200, 210, 190, 222, 199]})

record = pd.Series(['S6', 'Scarlette Fisher', 205], index=['student_id', 'name', 'marks'])

print("Original DataFrames:")
print(student_data1)
print("\nDictionary:")
print(record)

Original DataFrames:
  student_id              name  marks
0         S1  Danniella Fenton    200
1         S2      Ryder Storey    210
2         S3      Bryce Jensen    190
3         S4         Ed Bernal    222
4         S5       Kwame Morin    199

Dictionary:
student_id                  S6
name          Scarlette Fisher
marks                      205
dtype: object


In [35]:
#### your answer here
dicts = [{'student_id': 'S6', 'name': 'Scarlette Fisher', 'marks': 203}]
student_data2 = pd.DataFrame(dicts)
combined_data =  pd.concat([student_data1, student_data2], ignore_index=True)

print("\nCombined Data:")
print(combined_data)


Combined Data:
  student_id              name  marks
0         S1  Danniella Fenton    200
1         S2      Ryder Storey    210
2         S3      Bryce Jensen    190
3         S4         Ed Bernal    222
4         S5       Kwame Morin    199
5         S6  Scarlette Fisher    203


##### Exercise 2
(1) Concatenate the two given dataframes 'student_data1' and 'student_data2' along columns and store it as 'df1'.

(2) For 'student_data1', set 'name' as the row index and save it as 'index_data1'.

(3) Merge the dataframes 'index_data1' and 'student_data2' using student names, keep the union of all student data, remove any id relevant columns and store it as 'df2'.

(4) Reindex 'df2' starting from 0 and save it as 'df3'.

(5) Impute/Replace missing values in 'df3' with 0 and save it as 'df4'.

In [36]:
import pandas as pd
student_data1 = pd.DataFrame({
        'studentid': ['S1', 'S2', 'S3', 'S4', 'S5'],
         'name': ['Dante Morse', 'Ryder Storey', 'Bryce Jensen', 'Ed Bernal', 'Kwame Morin'],
        'history_marks': [200, 210, 190, 222, 199]})

student_data2 = pd.DataFrame({
        'student_id': ['S6', 'S4', 'S1', 'S9', 'S10'],
        'name': ['Scarlette Fisher', 'Ed Bernal', 'Dante Morse', 'Kaiser William', 'Madeeha Preston'],
        'math_marks': [201, 200, 198, 219, 201]})

print("Original DataFrames:")
print(student_data1)
print(student_data2)

Original DataFrames:
  studentid          name  history_marks
0        S1   Dante Morse            200
1        S2  Ryder Storey            210
2        S3  Bryce Jensen            190
3        S4     Ed Bernal            222
4        S5   Kwame Morin            199
  student_id              name  math_marks
0         S6  Scarlette Fisher         201
1         S4         Ed Bernal         200
2         S1       Dante Morse         198
3         S9    Kaiser William         219
4        S10   Madeeha Preston         201


In [42]:
## (1) (undesired outcome)
#### your answer here
df1 = pd.concat([student_data1, student_data2],axis=1)
print(df1)

  studentid          name  history_marks student_id              name  \
0        S1   Dante Morse            200         S6  Scarlette Fisher   
1        S2  Ryder Storey            210         S4         Ed Bernal   
2        S3  Bryce Jensen            190         S1       Dante Morse   
3        S4     Ed Bernal            222         S9    Kaiser William   
4        S5   Kwame Morin            199        S10   Madeeha Preston   

   math_marks  
0         201  
1         200  
2         198  
3         219  
4         201  


In [38]:
## (2)
#### your answer here
index_data1 = student_data1.set_index('name')
print(index_data1)

             studentid  history_marks
name                                 
Dante Morse         S1            200
Ryder Storey        S2            210
Bryce Jensen        S3            190
Ed Bernal           S4            222
Kwame Morin         S5            199


In [39]:
## (3)
#### your answer here
df2 = pd.merge(student_data2, index_data1, left_on='name', right_index=True, how = 'outer')
df2 = df2.drop(['student_id','studentid'],axis=1)
print(df2)

                 name  math_marks  history_marks
NaN      Bryce Jensen         NaN          190.0
2.0       Dante Morse       198.0          200.0
1.0         Ed Bernal       200.0          222.0
3.0    Kaiser William       219.0            NaN
NaN       Kwame Morin         NaN          199.0
4.0   Madeeha Preston       201.0            NaN
NaN      Ryder Storey         NaN          210.0
0.0  Scarlette Fisher       201.0            NaN


In [40]:
## (4)
#### your answer here
df3 = df2.reset_index(drop=True)
print(df3)

               name  math_marks  history_marks
0      Bryce Jensen         NaN          190.0
1       Dante Morse       198.0          200.0
2         Ed Bernal       200.0          222.0
3    Kaiser William       219.0            NaN
4       Kwame Morin         NaN          199.0
5   Madeeha Preston       201.0            NaN
6      Ryder Storey         NaN          210.0
7  Scarlette Fisher       201.0            NaN


In [41]:
## (5)
#### your answer here
df4 = df3.fillna(0)
print(df4)

               name  math_marks  history_marks
0      Bryce Jensen         0.0          190.0
1       Dante Morse       198.0          200.0
2         Ed Bernal       200.0          222.0
3    Kaiser William       219.0            0.0
4       Kwame Morin         0.0          199.0
5   Madeeha Preston       201.0            0.0
6      Ryder Storey         0.0          210.0
7  Scarlette Fisher       201.0            0.0


### GroupBy: Conditional Aggregation

In [43]:
import pandas as pd
import numpy as np
np.random.seed(1234)
df = pd.DataFrame({'key': ['A', 'B', 'C', 'A', 'B', 'C'],
                   'data': range(6),
                   'random': np.random.random(6)}, columns=['key', 'data', 'random'])
df

,key,data,random
0,A,0,0.191519
1,B,1,0.622109
2,C,2,0.437728
3,A,3,0.785359
4,B,4,0.779976
5,C,5,0.272593


In [44]:
# DataFrameGroupBy object: group data by the desired key column
df.groupby('key')

In [45]:
print(df.groupby('key').sum(),'\n')

     data    random
key                
A       3  0.976878
B       5  1.402085
C       7  0.710320 



In [46]:
print(df.groupby('key')['random'].sum())

key
A    0.976878
B    1.402085
C    0.710320
Name: random, dtype: float64


In [47]:
print(df.groupby('key').min())

     data    random
key                
A       0  0.191519
B       1  0.622109
C       2  0.272593


In [48]:
# iteration over groups
for (key, group) in df.groupby('key'):
    print((key,group),'\n')
for (key, group) in df.groupby('key'):
    print("{} shape={}".format(key, group.shape))

('A',   key  data    random
0   A     0  0.191519
3   A     3  0.785359) 

('B',   key  data    random
1   B     1  0.622109
4   B     4  0.779976) 

('C',   key  data    random
2   C     2  0.437728
5   C     5  0.272593) 

A shape=(2, 3)
B shape=(2, 3)
C shape=(2, 3)


In [49]:
# describe()
df.groupby('key')['random'].describe()

,count,mean,std,min,25%,50%,75%,max
key,,,,,,,,
A,2.0,0.488439,0.419908,0.191519,0.339979,0.488439,0.636899,0.785359
B,2.0,0.701042,0.111629,0.622109,0.661576,0.701042,0.740509,0.779976
C,2.0,0.355160,0.116768,0.272593,0.313876,0.355160,0.396444,0.437728


#### Aggregate, filter, transform, apply

In [50]:
#take a string, a function, or a list
df.groupby('key').aggregate(['min', np.median, max])

<ipython-input-50-57d9ff430fff>:2: FutureWarning: The provided callable <function median at 0x7dd6b8710700> is currently using SeriesGroupBy.median. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "median" instead.
  df.groupby('key').aggregate(['min', np.median, max])
<ipython-input-50-57d9ff430fff>:2: FutureWarning: The provided callable <built-in function max> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  df.groupby('key').aggregate(['min', np.median, max])


data               random                    
     min median max       min    median       max
key                                              
A      0    1.5   3  0.191519  0.488439  0.785359
B      1    2.5   4  0.622109  0.701042  0.779976
C      2    3.5   5  0.272593  0.355160  0.437728

In [51]:
#dictionary mapping
df.groupby('key').aggregate({'data': 'min',
                             'random': 'max'})

,data,random
key,,
A,0,0.785359
B,1,0.779976
C,2,0.437728


In [52]:
#filtering
def filter_func(x):
    return x['random'].min() < 0.3
print(df, '\n');
print(df.groupby('key').min(), '\n')

  key  data    random
0   A     0  0.191519
1   B     1  0.622109
2   C     2  0.437728
3   A     3  0.785359
4   B     4  0.779976
5   C     5  0.272593 

     data    random
key                
A       0  0.191519
B       1  0.622109
C       2  0.272593 



In [53]:
#keep groups that meet certain criteria
print(df.groupby('key').filter(filter_func))

  key  data    random
0   A     0  0.191519
2   C     2  0.437728
3   A     3  0.785359
5   C     5  0.272593


In [54]:
# transformation
# example:center the data by subtracting the group-wise mean
df.groupby('key').transform(lambda x: x - x.mean())

,data,random
0,-1.5,-0.296920
1,-1.5,-0.078934
2,-1.5,0.082568
3,1.5,0.296920
4,1.5,0.078934
5,1.5,-0.082568


In [56]:
# apply a function to the group results
def norm_by_data2(x):
    # x is a DataFrame of group values
    x['random'] /= x['data'].sum()
    return x

print(df,'\n'); print(df.groupby(df['key'], group_keys=False).apply(norm_by_data2)) #group data or not

  key  data    random
0   A     0  0.191519
1   B     1  0.622109
2   C     2  0.437728
3   A     3  0.785359
4   B     4  0.779976
5   C     5  0.272593 

  key  data    random
0   A     0  0.063840
1   B     1  0.124422
2   C     2  0.062533
3   A     3  0.261786
4   B     4  0.155995
5   C     5  0.038942


<ipython-input-56-89c4e750d8b5>:7: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  print(df,'\n'); print(df.groupby(df['key'], group_keys=False).apply(norm_by_data2)) #group data or not


#### Specification of the split key

In [57]:
# group data by a specified list
L = [2, 0, 0, 0, 1, 1]
print(df,'\n'); print(df.groupby(L).sum(numeric_only=True))

  key  data    random
0   A     0  0.191519
1   B     1  0.622109
2   C     2  0.437728
3   A     3  0.785359
4   B     4  0.779976
5   C     5  0.272593 

   data    random
0     6  1.845195
1     9  1.052568
2     0  0.191519


In [58]:
#group data by mapping
df2 = df.set_index('key')
mapping = {'A': 'vowel', 'B': 'consonant', 'C': 'consonant'}
print(df2,'\n'); print(df2.groupby(mapping).sum())

     data    random
key                
A       0  0.191519
B       1  0.622109
C       2  0.437728
A       3  0.785359
B       4  0.779976
C       5  0.272593 

           data    random
key                      
consonant    12  2.112405
vowel         3  0.976878


In [59]:
#group data by function
print(df2,'\n'); print(df2.groupby(str.lower).mean())

     data    random
key                
A       0  0.191519
B       1  0.622109
C       2  0.437728
A       3  0.785359
B       4  0.779976
C       5  0.272593 

     data    random
key                
a     1.5  0.488439
b     2.5  0.701042
c     3.5  0.355160


In [60]:
#group data by multi-index
df3 = df2.groupby([str.lower, mapping]).mean()
df3

,,data,random
key,key,,
a,vowel,1.5,0.488439
b,consonant,2.5,0.701042
c,consonant,3.5,0.355160


In [61]:
df3.index

MultiIndex([('a',     'vowel'),
            ('b', 'consonant'),
            ('c', 'consonant')],
           names=['key', 'key'])

##### Exercise 3
(1) Split the following dataframe into groups based on school code and class. Obtain mean, min, and max value of age for each group.

(2) Consider only the schools that have student age <= 13. Split the dataframe into groups based on school code, and obtain the average of BMI for each school. BMI: body mass index is defined as weight/(height/100)^2.

In [73]:
import pandas as pd
pd.set_option('display.max_rows', None)
#pd.set_option('display.max_columns', None)
student_data = pd.DataFrame({
    'school_code': ['s001','s002','s003','s001','s002','s004'],
    'class': ['V', 'V', 'VI', 'VI', 'V', 'VI'],
    'name': ['Alberto Franco','Gino Mcneill','Ryan Parkes', 'Eesha Hinton', 'Gino Mcneill', 'David Parkes'],
    'date_Of_Birth ': ['15/05/2002','17/05/2002','16/02/1999','25/09/1998','11/05/2002','15/09/1997'],
    'age': [12, 12, 13, 13, 14, 12],
    'height': [173, 192, 186, 167, 151, 159],
    'weight': [70, 85, 93, 65, 50, 60],
    'address': ['street1', 'street2', 'street3', 'street1', 'street2', 'street4']},
    index=['S1', 'S2', 'S3', 'S4', 'S5', 'S6'])
print("Original DataFrame:")
print(student_data)

Original DataFrame:
   school_code class            name date_Of_Birth   age  height  weight  \
S1        s001     V  Alberto Franco     15/05/2002   12     173      70   
S2        s002     V    Gino Mcneill     17/05/2002   12     192      85   
S3        s003    VI     Ryan Parkes     16/02/1999   13     186      93   
S4        s001    VI    Eesha Hinton     25/09/1998   13     167      65   
S5        s002     V    Gino Mcneill     11/05/2002   14     151      50   
S6        s004    VI    David Parkes     15/09/1997   12     159      60   

    address  
S1  street1  
S2  street2  
S3  street3  
S4  street1  
S5  street2  
S6  street4  


In [74]:
## (1)
#### your answer here
result = student_data.groupby(['school_code', 'class']).agg({'age': ['mean', 'min', 'max']})
print(result)

                    age        
                   mean min max
school_code class              
s001        V      12.0  12  12
            VI     13.0  13  13
s002        V      13.0  12  14
s003        VI     13.0  13  13
s004        VI     12.0  12  12


In [75]:
## (2)
#### your answer here
student_data['BMI'] = student_data['weight']/(student_data['height']/100)/(student_data['height']/100)
def filter_func(x):
    return x['age'].max() <= 13
df2 = student_data.groupby('school_code').filter(filter_func) ##keep the schools with student age <= 13
results = df2.groupby('school_code').agg({'BMI': 'mean'}) ##calculate school-level BMI average
print(results)

                   BMI
school_code           
s001         23.347683
s003         26.881720
s004         23.733238


### Pivot Table

In [62]:
# example dataset
import numpy as np
import pandas as pd
import seaborn as sns
titanic = sns.load_dataset('titanic')
titanic.head(6)

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True
5,0,3,male,NaN,0,0,8.4583,Q,Third,man,True,NaN,Queenstown,no,True


In [63]:
# group by class and gender
# select survival, apply a mean aggregate
# unstack the hierarchical index
titanic.groupby(['sex', 'class'])['survived'].aggregate('mean')

<ipython-input-63-ae42a141b027>:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  titanic.groupby(['sex', 'class'])['survived'].aggregate('mean')


sex     class 
female  First     0.968085
        Second    0.921053
        Third     0.500000
male    First     0.368852
        Second    0.157407
        Third     0.135447
Name: survived, dtype: float64

In [64]:
titanic.groupby(['sex', 'class'])['survived']

<ipython-input-64-a3bd334ce078>:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  titanic.groupby(['sex', 'class'])['survived']


In [65]:
titanic.groupby(['sex', 'class'])['survived'].aggregate('mean').unstack()

<ipython-input-65-45061042d882>:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  titanic.groupby(['sex', 'class'])['survived'].aggregate('mean').unstack()


class,First,Second,Third
sex,,,
female,0.968085,0.921053,0.500000
male,0.368852,0.157407,0.135447


In [66]:
#pivot table alternative
titanic.pivot_table('survived', index='sex', columns='class')

<ipython-input-66-b1b210a7eeee>:2: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  titanic.pivot_table('survived', index='sex', columns='class')


class,First,Second,Third
sex,,,
female,0.968085,0.921053,0.500000
male,0.368852,0.157407,0.135447


In [67]:
#multilevel pivot tables
#a third dimension, as an example
age = pd.cut(titanic['age'], [0, 18, 80])
titanic.pivot_table('survived', ['sex', age], 'class')

<ipython-input-67-834a848c8c94>:4: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  titanic.pivot_table('survived', ['sex', age], 'class')


class               First    Second     Third
sex    age                                   
female (0, 18]   0.909091  1.000000  0.511628
       (18, 80]  0.972973  0.900000  0.423729
male   (0, 18]   0.800000  0.600000  0.215686
       (18, 80]  0.375000  0.071429  0.133663

In [68]:
# multilevel at columns
fare = pd.qcut(titanic['fare'], 2)
titanic.pivot_table('survived', ['sex', age], [fare, 'class'])

<ipython-input-68-7f6f380ec76e>:3: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  titanic.pivot_table('survived', ['sex', age], [fare, 'class'])


fare            (-0.001, 14.454]                     (14.454, 512.329]  \
class                      First    Second     Third             First   
sex    age                                                               
female (0, 18]               NaN  1.000000  0.714286          0.909091   
       (18, 80]              NaN  0.880000  0.444444          0.972973   
male   (0, 18]               NaN  0.000000  0.260870          0.800000   
       (18, 80]              0.0  0.098039  0.125000          0.391304   

fare                                 
class              Second     Third  
sex    age                           
female (0, 18]   1.000000  0.318182  
       (18, 80]  0.914286  0.391304  
male   (0, 18]   0.818182  0.178571  
       (18, 80]  0.030303  0.192308

In [69]:
pd.DataFrame.pivot_table?

In [70]:
#check on the quantiles
titanic['fare'].quantile(0.5)

14.4542

In [71]:
#aggfunc: controls what type of aggregation is applied
titanic.pivot_table(index='sex', columns='class',
                    aggfunc={'survived':sum, 'fare':'mean'})

<ipython-input-71-9e76779a0339>:2: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  titanic.pivot_table(index='sex', columns='class',
<ipython-input-71-9e76779a0339>:2: FutureWarning: The provided callable <built-in function sum> is currently using SeriesGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  titanic.pivot_table(index='sex', columns='class',


fare                       survived             
class        First     Second      Third    First Second Third
sex                                                           
female  106.125798  21.970121  16.118810       91     70    72
male     67.226127  19.741782  12.661633       45     17    47

In [72]:
#margins
titanic.pivot_table('survived', index='sex', columns='class',
                    margins=True)

<ipython-input-72-a03fd62c44fc>:2: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  titanic.pivot_table('survived', index='sex', columns='class',


class,First,Second,Third,All
sex,,,,
female,0.968085,0.921053,0.500000,0.742038
male,0.368852,0.157407,0.135447,0.188908
All,0.629630,0.472826,0.242363,0.383838


##### Exercise 4

For the titanic data, find survival rate by gender, age of the different categories (0,18) (18,40) (40,65) (65,80) of various classes. Row: gender and age categories; column: classes.

In [76]:
#### your answer here
age = pd.cut(titanic['age'], [0, 18, 40, 65, 80])
result = titanic.pivot_table('survived', index=['sex', age], columns='class')
print(result)

class               First    Second     Third
sex    age                                   
female (0, 18]   0.909091  1.000000  0.511628
       (18, 40]  0.979167  0.914894  0.480000
       (40, 65]  0.961538  0.846154  0.111111
male   (0, 18]   0.800000  0.600000  0.215686
       (18, 40]  0.478261  0.063492  0.146199
       (40, 65]  0.282609  0.105263  0.068966
       (65, 80]  0.250000  0.000000  0.000000


<ipython-input-76-53a5705019cf>:3: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  result = titanic.pivot_table('survived', index=['sex', age], columns='class')


### String Operations

In [ ]:
# vectorized operation for numpy
data = ['peter', 'Paul', 'MARY', 'gUIDO']
[s.capitalize() for s in data]

In [ ]:
data = ['peter', 'Paul', None, 'MARY', 'gUIDO']
[s.capitalize() for s in data]

In [ ]:
# pandas is convenient
import pandas as pd
names = pd.Series(data)
names.str.capitalize()

In [ ]:
print(names,'\n')
print(names.str.upper(),'\n')
print(names.str.swapcase())

#### String methods available:
len() lower() translate() islower() <br>
ljust() upper() startswith() isupper() <br>
rjust() find() endswith() isnumeric() <br>
center() rfind() isalnum() isdecimal() <br>
zfill() index() isalpha() split() <br>
strip() rindex() isdigit() rsplit() <br>
rstrip() capitalize() isspace() partition() <br>
lstrip() swapcase() istitle() rpartition() <br>

In [ ]:
monte = pd.Series(['Graham Chapman', 'John Cleese', 'Terry Gilliam',
'Eric Idle', 'Terry Jones', 'Michael Palin'])
monte.str.len()

In [ ]:
monte.str.startswith('T')

In [ ]:
pd.Series.str.split?
monte.str.split()

#### Miscellaneous methods
get() Index each element <br>
slice() Slice each element<br>
slice_replace() Replace slice in each element with passed value<br>
cat() Concatenate strings<br>
repeat() Repeat values<br>
normalize() Return Unicode form of string<br>
pad() Add whitespace to left, right, or both sides of strings<br>
wrap() Split long strings into lines with length less than a given width<br>
join() Join strings in each element of the Series with passed separator<br>
get_dummies() Extract dummy variables as a DataFrame

In [ ]:
#vectorized element access
monte.str[0:3]

In [ ]:
#last element of each entry
monte.str.split().str.get(-1)

In [ ]:
full_monte = pd.DataFrame({'name': monte,
                           'info': ['B|C|D', 'B|D', 'A|C',
                                    'B|D', 'B|C','B|C|D']})
full_monte

In [ ]:
#quickly split out these indicator variables into a DataFrame
full_monte['info'].str.get_dummies('|')